# CC2017 – Modelación y Simulación
## Laboratorio 5: Task 3 (Entrega Final) – Isocronas, revisión de literatura y análisis crítico
**Territorio:** Texas (TX), igual que el Task 2 · **CRS métrico:** EPSG:3081 · **Datos:** `data/` (estados, condados, hospitales, población)

| Sección | Contenido |
|---|---|
| 3.1 | Red vial con OSMnx, isocronas de 15 min vs. buffer de 10 km, argumento formal e implicaciones de política |
| 3.2 | Paper 2022–2026 (Nairobi, Kenia): cita APA, comparación metodológica y de brechas |
| 3.3 | Sensibilidad del índice de vulnerabilidad, caso adverso del algoritmo voraz y limitaciones para el ministerio |

> **Requisitos:** `pip install geopandas osmnx networkx scipy matplotlib`. La sección 3.1 **descarga datos de OpenStreetMap** (Overpass API), por lo que necesita conexión a internet; los grafos se guardan en `cache_osm/` y no se vuelven a descargar.

## 0. Preparación: datos del Task 2 y recálculo del índice de vulnerabilidad

In [ ]:
# !pip -q install geopandas osmnx networkx scipy matplotlib
import sys, os, re, warnings
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, shapely, osmnx as ox
from shapely.ops import unary_union
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
warnings.filterwarnings("ignore", category=UserWarning)

print("GeoPandas:", gpd.__version__, "| Shapely:", shapely.__version__, "| OSMnx:", ox.__version__, "| NetworkX:", nx.__version__)

In [ ]:
# --- Datos de Texas (idéntico al Task 2) ---
DATA_DIR = Path("data")
ESTADO_COD, ESTADO_FIPS, CRS_METRICO = "TX", "48", "EPSG:3081"

estados_raw    = gpd.read_file(DATA_DIR / "estados_eeuu.geojson")
condados_raw   = gpd.read_file(DATA_DIR / "condados_eeuu.geojson")
hospitales_raw = gpd.read_file(DATA_DIR / "hospitales_eeuu.geojson")
pob_raw        = pd.read_csv(DATA_DIR / "poblacion_condados.csv", dtype={"fips": str})

pob = pob_raw[~pob_raw["fips"].str[:2].isin({"80", "90"})].dropna(subset=["poblacion"]).copy()
pob["fips"] = pob["fips"].str.zfill(5)
hosp_clean = hospitales_raw.dropna(subset=["camas_total"]).copy()

estado_gdf = estados_raw[estados_raw["codigo"] == ESTADO_COD].to_crs(CRS_METRICO).reset_index(drop=True)
limite_estado = estado_gdf.geometry.iloc[0]

cond = condados_raw[condados_raw["cod_estado"] == ESTADO_FIPS].merge(pob[["fips", "poblacion"]], on="fips", how="left")
cond["poblacion"] = cond["poblacion"].fillna(0)
cond = cond.to_crs(CRS_METRICO).reset_index(drop=True)
cond["area_m2"] = cond.area
hosp_tx = hosp_clean[hosp_clean["estado"] == ESTADO_COD].to_crs(CRS_METRICO).reset_index(drop=True)
print(f"Condados: {len(cond)} | Hospitales con camas: {len(hosp_tx)} | Población: {cond['poblacion'].sum():,.0f}")

In [ ]:
# --- Componentes C1, C2, C3 y el índice base (Task 2.2) ---
cxy = np.c_[cond.geometry.centroid.x, cond.geometry.centroid.y]
hxy = np.c_[hosp_tx.geometry.x, hosp_tx.geometry.y]
tree_hosp = cKDTree(hxy)
d, idx_cercano = tree_hosp.query(cxy)
cond["dist_km"] = d / 1000.0
cond["idx_hosp_cercano"] = idx_cercano

minmax = lambda s: (s - s.min()) / (s.max() - s.min())
cond["C1"] = minmax(cond["dist_km"])

pin = gpd.sjoin(hosp_tx, cond[["fips", "geometry"]], how="inner", predicate="within")   # hospital -> condado
cond["camas_cond"] = cond["fips"].map(pin.groupby("fips")["camas_total"].sum()).fillna(0)
cpc = np.where(cond["poblacion"] > 0, cond["camas_cond"] / cond["poblacion"], 0.0)
tiene = cpc > 0
inv = pd.Series(np.nan, index=cond.index); inv[tiene] = 1.0 / cpc[tiene]; inv[~tiene] = inv[tiene].max()   # sin hospital -> máximo
cond["C2"] = minmax(inv)

oc = []
for ii in tree_hosp.query_ball_point(cxy, r=50_000):
    v = hosp_tx.iloc[ii]["ocupacion_total"].dropna() if len(ii) else []
    oc.append(np.nan if len(v) == 0 else v.mean())
c3 = pd.Series(oc, index=cond.index); c3 = c3.fillna(c3.dropna().max())                              # sin hospital a 50 km -> máximo
cond["C3"] = minmax(c3)

W_BASE = np.array([0.40, 0.35, 0.25])
cond["Iv"] = cond[["C1", "C2", "C3"]].values @ W_BASE
display(cond.sort_values("Iv", ascending=False).head(10)[["fips", "nombre", "poblacion", "dist_km", "C2", "C3", "Iv"]].round(3))
print("Condados sin hospital en su territorio:", int((~tiene).sum()), "de", len(cond))

---
## Task 3.1 – Buffer circular vs. isocrona de tiempo de viaje
### 3.1a Selección de las tres instalaciones

**Hallazgo previo:** los 10 condados más vulnerables del Task 2.2 (Terrell, Jim Hogg, Edwards, …) **no contienen ningún hospital** (para todos $C_2=C_3=1$ por la regla de imputación del máximo). Por eso "las instalaciones ubicadas en las divisiones de mayor vulnerabilidad" admite dos lecturas, ambas implementadas con `MODO_HOSPITALES`:

* `"dentro"` *(por defecto, lectura literal)*: el hospital de mayor capacidad ubicado en cada uno de los **3 condados más vulnerables que sí tienen hospital**.
* `"servidor"`: el hospital más cercano a cada uno de los **3 condados más vulnerables** (los que realmente los atienden: Iraan General, Starr County Memorial, Lillian M. Hudspeth).

In [ ]:
MODO_HOSPITALES = "dentro"      # "dentro" | "servidor"

if MODO_HOSPITALES == "dentro":
    top = cond[cond["camas_cond"] > 0].sort_values("Iv", ascending=False).head(3)
    filas = []
    for _, r in top.iterrows():
        h = pin[pin["fips"] == r["fips"]].sort_values("camas_total", ascending=False).iloc[0]
        filas.append((r["nombre"], r["Iv"], h))
else:
    top = cond.sort_values("Iv", ascending=False).head(3)
    filas = [(r["nombre"], r["Iv"], hosp_tx.iloc[int(r["idx_hosp_cercano"])]) for _, r in top.iterrows()]

sel = gpd.GeoDataFrame(
    [{"condado_vulnerable": c, "Iv": iv, "hospital": h["nombre"], "ciudad": h["ciudad"], "tipo": h["tipo"],
      "camas": h["camas_total"], "geometry": h.geometry} for c, iv, h in filas], crs=CRS_METRICO)
ll = sel.to_crs(4326).geometry
sel["lon"], sel["lat"] = ll.x, ll.y
sel["slug"] = [re.sub(r"\W+", "_", n.lower()) for n in sel["hospital"]]
display(sel.drop(columns="geometry").round(3))

### 3.1b Red vial (OSMnx), velocidades por tipo de vía y tiempos de viaje

Texas completo es demasiado grande para una sola descarga (cientos de miles de km de vías), así que se descarga una **subregión por hospital**: la red `drive` en un cuadro de $\pm40$ km alrededor del hospital. A 15 min, incluso a 120 km/h, el alcance máximo es ≈34 km (85 mph), por lo que el cuadro contiene la isocrona; el código avisa si el alcance toca el borde (truncamiento).

**Velocidades:** si OSM trae `maxspeed` se usa; si no, se imputa por tipo de vía con la tabla `VEL_KMH` (flujo libre, **sin congestión**). Tiempo por arista: $t_e=\dfrac{\ell_e}{v_e}$.

In [ ]:
CACHE = Path("cache_osm"); CACHE.mkdir(exist_ok=True)
RADIO_DESCARGA_M = 40_000

# Velocidades de flujo libre (km/h) para aristas sin 'maxspeed' en OSM
VEL_KMH = {"motorway": 110, "motorway_link": 65, "trunk": 100, "trunk_link": 60, "primary": 90, "primary_link": 50,
           "secondary": 80, "secondary_link": 45, "tertiary": 65, "tertiary_link": 40,
           "unclassified": 50, "residential": 35, "living_street": 15, "service": 25}

def grafo_hospital(lat, lon, slug):
    """Descarga (o carga de caché) la red vial en coche alrededor del hospital y la proyecta a CRS_METRICO."""
    ruta = CACHE / f"{slug}.graphml"
    if ruta.exists():
        G = ox.load_graphml(ruta)
    else:
        G = ox.graph_from_point((lat, lon), dist=RADIO_DESCARGA_M, network_type="drive", simplify=True)
        ox.save_graphml(G, ruta)
    G = ox.add_edge_speeds(G, hwy_speeds=VEL_KMH, fallback=50)     # km/h
    G = ox.add_edge_travel_times(G)                                # segundos
    return ox.project_graph(G, to_crs=CRS_METRICO)

grafos = {}
for _, r in sel.iterrows():
    grafos[r["slug"]] = grafo_hospital(r["lat"], r["lon"], r["slug"])
    G = grafos[r["slug"]]
    print(f"{r['hospital']:<45s} nodos={G.number_of_nodes():>6,} aristas={G.number_of_edges():>6,}")

### 3.1c Isocrona de 15 min y comparación con el buffer de 10 km

La isocrona se construye como el conjunto de nodos alcanzables con $T(f,x)\le 15$ min (Dijkstra sobre `travel_time`) y la unión de sus aristas con un **ancho de influencia** de 1 km (área que se considera atendida a ambos lados de la vía). Ese ancho es una decisión de modelado: más abajo se reporta la sensibilidad a 0.5, 1 y 2 km.

Cobertura poblacional con el mismo criterio ponderado por área del Task 1.3b/2.1b:
$$P_{\text{cub}}(Z)=\sum_c P_c\,\frac{A(c\cap Z)}{A(c)},\qquad Z\in\{\text{Buffer}_{10\,\text{km}},\ \text{Iso}_{15\,\text{min}}\}$$

In [ ]:
T_MIN, R_BUFFER_M, ANCHO_ISO_M = 15, 10_000, 1000

def isocrona(Gp, xy0, t_min=T_MIN, ancho_m=ANCHO_ISO_M):
    """Isocrona de t_min minutos: polígono + tabla de nodos alcanzados (tiempo, distancia recta, velocidad efectiva)."""
    n0 = ox.distance.nearest_nodes(Gp, xy0[0], xy0[1])
    pred, tt = nx.dijkstra_predecessor_and_distance(Gp, n0, cutoff=t_min * 60, weight="travel_time")
    nodos = pd.DataFrame({"t_s": pd.Series(tt)})
    nodos["x"] = [Gp.nodes[n]["x"] for n in nodos.index]
    nodos["y"] = [Gp.nodes[n]["y"] for n in nodos.index]
    nodos["recta_m"] = np.hypot(nodos["x"] - xy0[0], nodos["y"] - xy0[1])
    # velocidad efectiva = progreso en línea recta / tiempo (km/h); captura velocidad de la vía Y rodeos de la red
    nodos["v_ef_kmh"] = np.where(nodos["t_s"] > 0, nodos["recta_m"] / 1000 / (nodos["t_s"] / 3600), np.nan)
    aristas = ox.graph_to_gdfs(Gp, nodes=False)
    u = aristas.index.get_level_values(0); v = aristas.index.get_level_values(1)
    alc = aristas[u.isin(nodos.index) & v.isin(nodos.index)]
    poly = unary_union(alc.geometry.buffer(ancho_m)).intersection(limite_estado)
    return poly, nodos, alc

def pob_en(poly):
    """Población ponderada por área dentro de poly (Task 1.3b / 2.1b)."""
    ii = cond.sindex.query(poly, predicate="intersects")
    sub = cond.iloc[ii]
    frac = (sub.geometry.intersection(poly).area / sub["area_m2"]).clip(0, 1)
    return float((sub["poblacion"] * frac).sum())

res, geoms = [], {}
for _, r in sel.iterrows():
    Gp = grafos[r["slug"]]
    xy0 = (r.geometry.x, r.geometry.y)
    iso, nodos, alc = isocrona(Gp, xy0)
    buf = r.geometry.buffer(R_BUFFER_M).intersection(limite_estado)
    geoms[r["slug"]] = dict(iso=iso, buf=buf, nodos=nodos, alc=alc)

    if nodos["recta_m"].max() > 0.9 * RADIO_DESCARGA_M:
        print(f"⚠ {r['hospital']}: la isocrona toca el borde del grafo; aumenta RADIO_DESCARGA_M.")
    sobre, sub_ = buf.difference(iso), iso.difference(buf)
    res.append({"Hospital": r["hospital"], "Condado": r["condado_vulnerable"],
                "Área buffer 10 km (km²)": buf.area / 1e6, "Área isocrona 15 min (km²)": iso.area / 1e6,
                "Sobreestimada (buffer∖iso, km²)": sobre.area / 1e6, "Subestimada (iso∖buffer, km²)": sub_.area / 1e6,
                "Pob. buffer": pob_en(buf), "Pob. isocrona": pob_en(iso),
                "Pob. sobreestimada": pob_en(sobre), "Pob. subestimada": pob_en(sub_)})
tabla_31 = pd.DataFrame(res)
tabla_31["Δ Pob. (iso − buffer)"] = tabla_31["Pob. isocrona"] - tabla_31["Pob. buffer"]
tabla_31["Δ %"] = tabla_31["Δ Pob. (iso − buffer)"] / tabla_31["Pob. buffer"].replace(0, np.nan) * 100

# Cobertura conjunta de las tres instalaciones (unión)
buf_u = unary_union([g["buf"] for g in geoms.values()]); iso_u = unary_union([g["iso"] for g in geoms.values()])
tot = {"Hospital": "UNIÓN (3 instalaciones)", "Condado": "—",
       "Área buffer 10 km (km²)": buf_u.area/1e6, "Área isocrona 15 min (km²)": iso_u.area/1e6,
       "Sobreestimada (buffer∖iso, km²)": buf_u.difference(iso_u).area/1e6, "Subestimada (iso∖buffer, km²)": iso_u.difference(buf_u).area/1e6,
       "Pob. buffer": pob_en(buf_u), "Pob. isocrona": pob_en(iso_u),
       "Pob. sobreestimada": pob_en(buf_u.difference(iso_u)), "Pob. subestimada": pob_en(iso_u.difference(buf_u))}
tot["Δ Pob. (iso − buffer)"] = tot["Pob. isocrona"] - tot["Pob. buffer"]; tot["Δ %"] = tot["Δ Pob. (iso − buffer)"] / tot["Pob. buffer"] * 100
tabla_31 = pd.concat([tabla_31, pd.DataFrame([tot])], ignore_index=True)
display(tabla_31.round(1).set_index("Hospital").T)

In [ ]:
# Sensibilidad al ancho de influencia de la isocrona (decisión de modelado)
filas = []
for _, r in sel.iterrows():
    g = geoms[r["slug"]]
    for w in (500, 1000, 2000):
        p = unary_union(g["alc"].geometry.buffer(w)).intersection(limite_estado)
        filas.append({"Hospital": r["hospital"], "Ancho (m)": w, "Área iso (km²)": p.area/1e6, "Pob. iso": pob_en(p),
                      "Pob. buffer 10 km": pob_en(g["buf"])})
display(pd.DataFrame(filas).round(0).astype({"Ancho (m)": int}))

In [ ]:
# Mapas: buffer circular (azul) vs isocrona (naranja) y red alcanzada
def escala(ax, bounds, km=10):
    x0, y0, x1, y1 = bounds; xs, ys = x0 + .05*(x1-x0), y0 + .05*(y1-y0)
    ax.plot([xs, xs + km*1000], [ys, ys], "k-", lw=4); ax.text(xs + km*500, ys + .015*(y1-y0), f"{km} km", ha="center", fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), dpi=110)
for ax, (_, r) in zip(axes, sel.iterrows()):
    g = geoms[r["slug"]]
    cond.plot(ax=ax, color="#f5f5f5", edgecolor="#cccccc", lw=.3)
    gpd.GeoSeries([g["iso"]], crs=CRS_METRICO).plot(ax=ax, color="#fd8d3c", alpha=.45)
    g["alc"].plot(ax=ax, color="#7f2704", lw=.5)
    gpd.GeoSeries([g["buf"]], crs=CRS_METRICO).boundary.plot(ax=ax, color="#08519c", lw=2)
    gpd.GeoSeries([r.geometry], crs=CRS_METRICO).plot(ax=ax, color="red", marker="*", markersize=220, zorder=5)
    b = unary_union([g["buf"], g["iso"]]).buffer(3000).bounds; ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3])
    escala(ax, (b[0], b[1], b[2], b[3]))
    ax.set_title(f"{r['hospital'].title()}\n({r['condado_vulnerable']} Co.)", fontsize=10); ax.set_axis_off()
axes[0].legend(handles=[Patch(color="#fd8d3c", alpha=.5, label="Isocrona 15 min"), plt.Line2D([], [], color="#08519c", lw=2, label="Buffer 10 km"),
                        plt.Line2D([], [], color="#7f2704", lw=1, label="Red alcanzada")], loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Velocidad efectiva v_ef = ||x - f|| / T(f,x)  -> criterio formal de 3.1d (umbral = R/τ = 10 km / 0.25 h = 40 km/h)
UMBRAL = R_BUFFER_M / 1000 / (T_MIN / 60)
filas = []
for _, r in sel.iterrows():
    n = geoms[r["slug"]]["nodos"]; n = n[n["recta_m"] >= 2000]          # excluye el entorno inmediato
    filas.append({"Hospital": r["hospital"], "Nodos (≥2 km)": len(n),
                  f"% nodos con v_ef > {UMBRAL:.0f} km/h (buffer SUBestima)": (n["v_ef_kmh"] > UMBRAL).mean() * 100,
                  f"% nodos con v_ef < {UMBRAL:.0f} km/h (buffer SOBREestima)": (n["v_ef_kmh"] < UMBRAL).mean() * 100,
                  "v_ef mediana (km/h)": n["v_ef_kmh"].median(), "Alcance recto máx. (km)": n["recta_m"].max() / 1000})
display(pd.DataFrame(filas).round(1).set_index("Hospital").T)

### 3.1d Cuándo el buffer circular sobreestima y cuándo subestima

**Formalización.** Sea $f$ la instalación, $\tau=15$ min, $R=10$ km y $v_{ef}(x)=\|x-f\|/T(f,x)$ la *velocidad efectiva* hacia $x$ (progreso en línea recta por unidad de tiempo; incorpora tanto la velocidad de las vías como los rodeos, $v_{ef}=\bar v/c$ con $c=d_{red}/\|x-f\|\ge1$ el factor de circuidad). Entonces
$$x\in \text{Iso}_\tau \iff \|x-f\|\le v_{ef}(x)\,\tau,\qquad x\in\text{Buf}_R \iff \|x-f\|\le R .$$
En coordenadas polares la isocrona tiene alcance radial $\rho(\theta)=v_{ef}(\theta)\,\tau$ y el buffer alcance constante $R$:

* **Sobreestimación** ($x\in\text{Buf}\setminus\text{Iso}$) en las direcciones con $v_{ef}(\theta)<R/\tau=40$ km/h.
* **Subestimación** ($x\in\text{Iso}\setminus\text{Buf}$) en las direcciones con $v_{ef}(\theta)>40$ km/h.
* A nivel de área: $A_{\text{Iso}}=\tfrac12\int_0^{2\pi}\rho(\theta)^2\,d\theta$ frente a $A_{\text{Buf}}=\pi R^2$; el buffer es **correcto en promedio solo si** $\overline{\rho^2}=R^2$, y nunca lo es *direccionalmente* salvo que $v_{ef}$ sea isótropa. Los errores se compensan en el área pero **no en la población**, porque ésta no se distribuye isótropamente alrededor del hospital.

**Geometría de la red vial de Texas** (los indicadores de la tabla anterior cuantifican cada punto):

1. **Corredores de alta velocidad (subestimación).** La red de Texas es una malla de grandes autopistas radiales/transversales (I-10, I-20, I-35, I-45 y US/SH rurales) con límites de hasta 75–85 mph en tramos rurales (≈120–137 km/h). A $v_{ef}\approx100$ km/h, $\rho=25$ km $=2.5R$: a lo largo de un corredor la isocrona se extiende más del doble que el buffer, y el buffer **subestima** la cobertura de las localidades alineadas sobre esa vía.
2. **Red escasa y lineal en el oeste y el sur (sobreestimación transversal).** En condados como Brewster o Val Verde las vías son pocas, largas y casi paralelas a un solo eje (p. ej. US-90, US-67 o SH-118 en el Trans-Pecos); la tabla de 3.1c permite verificarlo con la velocidad efectiva de cada hospital. Fuera del corredor no hay vías (grado de nodo bajo, $c\to\infty$ o simplemente sin acceso), por lo que $v_{ef}(\theta)\approx0$ en las direcciones perpendiculares: la isocrona es una **cinta** y el buffer **sobreestima** el área cubierta en todo el resto del disco. Esto se agrava por barreras físicas: el Río Bravo (frontera internacional, con pocos puentes), y macizos sin vías (Big Bend, Davis Mountains), donde el disco de 10 km incluye terreno inaccesible.
3. **Malla urbana/periurbana densa (sobreestimación moderada o neutra).** Con calles de 25–45 mph ($40$–$72$ km/h) y $c\approx1.2$–$1.4$, $v_{ef}\approx 30$–$55$ km/h, cercano al umbral de 40 km/h: buffer e isocrona tienen áreas similares, pero el buffer **sobreestima** en cuanto se añade congestión (no modelada aquí) o ríos/ferrocarriles con pocos cruces.
4. **Efecto de borde de la unión de buffers.** La unión de discos solapados en una misma autopista sobreestima el área *nueva* que aporta cada hospital adicional, lo que afecta directamente el MCLP del Task 2.3.

**Conclusión:** el sesgo del buffer no tiene signo fijo; depende de si el hospital se ubica **sobre** un corredor rápido (subestima) o **en una zona de red escasa o fragmentada** (sobreestima). En el Texas rural remoto predominan los casos 1–2 simultáneamente: subestima a lo largo de la autopista y sobreestima fuera de ella.

### 3.1e Implicaciones para política pública

In [ ]:
for _, t in tabla_31.iterrows():
    signo = "MENOS" if t["Δ Pob. (iso − buffer)"] < 0 else "MÁS"
    print(f"• {t['Hospital']}: la isocrona de 15 min cubre {abs(t['Δ %']):.1f} % {signo} población que el buffer de 10 km "
          f"({t['Δ Pob. (iso − buffer)']:+,.0f} hab.); población 'cubierta solo en el mapa' = {t['Pob. sobreestimada']:,.0f}, "
          f"'cubierta realmente pero omitida' = {t['Pob. subestimada']:,.0f}.")

Usar el buffer circular en vez de la isocrona **no solo cambia un número**: cambia *qué* se decide. Al menos cuatro decisiones concretas cambiarían:

1. **Priorización geográfica de inversión (dónde se construye/refuerza).** Con buffers, un condado dentro de los 10 km de un hospital figura como cubierto y sale de la lista de prioridad; con isocronas, la población “cubierta solo en el mapa” (la que está en la zona $\text{Buf}\setminus\text{Iso}$ de la tabla anterior) vuelve a ser demanda no atendida. Dado que el índice de vulnerabilidad $C_1$ usa distancia euclidiana, el ranking de los condados en el borde del hospital cambiaría y el mismo presupuesto se asignaría a otros condados.
2. **Localización de nuevos hospitales (MCLP, Task 2.3).** El voraz maximiza población nueva cubierta por discos de 25 km. Con isocronas el beneficio marginal de un candidato depende de su acceso a una autopista: sitios en cruces de corredores ganan, y sitios geográficamente centrales pero sin acceso vial pierden. La solución del Task 2.3 (candidatos #164, #0 y #236) podría cambiar y la ganancia de 0.63/0.45/0.29 puntos porcentuales no es transferible.
3. **Despliegue de EMS, helipuertos y telemedicina.** Un estándar de respuesta se expresa en minutos, no en kilómetros. Si el buffer declara “cubierta” a una población que en realidad queda a más de 15 min, no se justificaría una base de ambulancias ni una ambulancia aérea; con isocronas, esa zona (Buf∖Iso) es la candidata natural, y la inversa (Iso∖Buf) permite *retirar* recursos redundantes a lo largo de autopistas.
4. **Elegibilidad y financiamiento de hospitales rurales (p. ej., Critical Access).** La regla de elegibilidad de los CAH de CMS se formula en **distancia por carretera** a otro hospital (35 millas; 15 millas en terreno montañoso o con solo vías secundarias; verifica la regla vigente), no en distancia euclidiana. Evaluar la red con buffers clasificaría mal los hospitales que están “cerca” en línea recta pero lejos por carretera (o al revés) y alteraría qué instalaciones califican para pagos diferenciales.

Un efecto transversal: con buffers el territorio parece **más homogéneo y mejor cubierto** de lo que es, lo que retrasa políticas de transporte (no emergente) y de coordinación entre sectores salud–transporte.

---
## Task 3.2 – Revisión de literatura
### 3.2a Referencia (APA 7) y caracterización

> Mutono, N., Wright, J. A., Mutunga, M., Mutembei, H., & Thumbi, S. M. (2022). Impact of traffic congestion on spatial access to healthcare services in Nairobi. *Frontiers in Health Services, 2*, Article 788173. https://doi.org/10.3389/frhs.2022.788173

*(País de ingreso medio-bajo: Kenia, clasificado por el Banco Mundial como de ingreso mediano-bajo. Verifica el paper en Scopus/Google Scholar y ajusta el formato si tu curso exige otra variante de APA.)*

| Aspecto | Paper |
|---|---|
| **Métrica de cobertura** | **Isocronas** de 15, 30 y 45 min en coche (picos y valles de tráfico) + **índice de accesibilidad 2SFCA** (razón instalaciones/población y profesionales/población, umbral OMS de 4.45 por 1 000) |
| **Red de servicios** | *Kenya Master Health Facility List* (Ministerio de Salud): 944 primarias, 94 secundarias y 4 terciarias en el condado de Nairobi; coordenadas de las públicas de una base previa (Maina et al., 2019) y de las privadas geocodificadas con Google Earth Pro |
| **Red vial / tiempos** | Servicio de ruteo de ESRI con datos históricos de **sondas de tráfico** (GPS) para lunes 8 am, miércoles 6 pm (pico) y sábado/domingo (valle) |
| **Población** | WorldPop 2019 (redistribución dasimétrica con *random forest* del censo de 2009) |
| **Resolución espacial** | Malla de **100 m × 100 m** |
| **Hallazgo principal** | En hora pico <70 % de los 4.1 millones de habitantes queda a ≤30 min de un establecimiento (>75 % en valle); la periferia es la peor atendida |

### 3.2b Comparación metodológica con este laboratorio

| Decisión | Este laboratorio | Mutono et al. (2022) | Efecto en la validez |
|---|---|---|---|
| **1. Medida de acceso** | Distancia euclidiana/buffers (Task 1–2) y, en 3.1, isocrona de flujo libre con velocidades por tipo de vía | Isocronas con **tráfico real** (sondas) y distinción pico/valle | **Mejora** la validez: el tiempo, no el kilómetro, es lo que determina el acceso; con una velocidad de flujo libre nuestra isocrona sigue *sobreestimando* en zonas congestionadas. |
| **2. Unidad demográfica** | Población **por condado**, supuesta uniforme dentro de cada polígono (ponderación por área) | Población en celdas de **100 m** (WorldPop) | **Mejora**: evita el sesgo de la unidad de área modificable (MAUP) y la falacia ecológica; nuestro supuesto de uniformidad es fuerte en condados extensos y despoblados. |
| **3. Oferta** | Hospital = punto con peso uniforme en la cobertura; capacidad solo entra en el índice $C_2$ | Estratifica por **nivel** (primario/secundario/terciario) y usa **2SFCA** (capacidad y competencia por demanda), más fuerza laboral | **Mejora**: un hospital de 12 camas y uno de 280 no cubren “igual”. Limitación del paper: usa promedios de personal por nivel (KSPA 2010) y supone acceso igual dentro del radio. |
| **4. Efecto de borde** | Se recorta a la frontera de Texas; no hay hospitales de otros estados ni de México | Control explícito con un **buffer de 2.5 km** más allá del límite administrativo | **Mejora**: ignorar hospitales cercanos fuera del límite subestima cobertura en condados fronterizos (nuestro caso). |
| **5. Alcance y reproducibilidad** | Estatal (254 condados), datos abiertos, código reproducible, optimización (MCLP) e índice de vulnerabilidad | Una sola ciudad-condado; datos de tráfico **propietarios** (ESRI) y 4 franjas horarias | **Limita** al paper: no es extrapolable a un territorio rural/extenso ni reproducible sin licencia; nuestro análisis es más escalable pero menos preciso por unidad. |

### 3.2c Comparación de la brecha de cobertura

In [ ]:
# Cifras del paper (Tabla 2, Nairobi, 4.1 M hab., nivel secundario = análogo "hospital") vs. Texas (Task 2.1b y 3.1)
paper = pd.DataFrame({"Umbral": ["15 min", "30 min", "45 min"],
                      "Pico (% cubierto)": [20, 62, 100], "Valle (% cubierto)": [43, 71, 100]})
paper["Brecha pico (%)"] = 100 - paper["Pico (% cubierto)"]; paper["Brecha valle (%)"] = 100 - paper["Valle (% cubierto)"]
print("Nairobi (Mutono et al., 2022), establecimientos secundarios:"); display(paper)

# Texas: curva buffer del Task 2.1b (valores del cuaderno Task2)
texas = pd.DataFrame({"Radio buffer (km)": [10, 20, 25, 30], "% cubierto": [46.91, 76.92, 86.85, 92.63]})
texas["Brecha (%)"] = 100 - texas["% cubierto"]; texas["Población sin cobertura"] = ((texas["Brecha (%)"]/100) * cond["poblacion"].sum()).round()
print("Texas (buffer euclidiano, Task 2.1b):"); display(texas)

**¿Es comparable en magnitud?** Solo con cautela, porque difieren la medida (tiempo vs. radio), el tipo de servicio (el paper incluye clínicas y dispensarios; nuestros datos son solo hospitales) y la escala (una ciudad de 4.1 M vs. un estado de 29 M). Tomando lo comparable (acceso a nivel hospital): en Nairobi, a 30 min **29–38 %** de la población queda sin acceso a un establecimiento secundario; en Texas a ≈25 km (≈20–25 min por autopista) la brecha es **13.2 %** (3.8 M de personas), y a 10 km es 53 %. La brecha *relativa* es menor en Texas, pero está **concentrada espacialmente** (oeste y sur, baja densidad), mientras que en Nairobi se concentra en la **periferia densa** y en la hora pico.

**Factores que explican similitudes y diferencias:**

* **Estructura geográfica.** Texas: población concentrada en pocas áreas metropolitanas (Houston, DFW, Austin, San Antonio) y un interior extenso y despoblado; la brecha es *de distancia*. Nairobi: ciudad compacta, la brecha es *de velocidad* (congestión y periferia en expansión no planificada).
* **Red y movilidad.** Texas tiene una malla de autopistas de alta velocidad y alta motorización, de modo que 25 km son ≈20 min; en Nairobi el uso de *matatus* y la congestión reducen la velocidad efectiva y hacen que el radio y el tiempo se desacoplen más (el paper muestra 3–23 puntos de diferencia pico–valle).
* **Estructura del sistema de salud.** Kenia: 74 % de las instalaciones son privadas, servicio devuelto a los condados desde 2013 y escasez de personal (<4.45 profesionales por 1 000 en primarios y secundarios). EE. UU./Texas: hospitales privados o sin fines de lucro, financiamiento y seguro condicionan la viabilidad de hospitales rurales pequeños (por eso los “Critical Access” de 12–25 camas del Task 2.1c).
* **Similitud clave:** en ambos casos **la cobertura “geográfica” sobrestima el acceso efectivo**: el paper ve que la capacidad (profesionales) y el tráfico reducen el acceso real, y en Texas los condados más alejados dependen de hospitales de 12–25 camas sin UCI.

---
## Task 3.3 – Reflexión sobre las decisiones de modelado
### 3.3a Análisis de sensibilidad del índice de vulnerabilidad

Se recalcula $I_v=w_1C_1+w_2C_2+w_3C_3$ con **seis configuraciones de pesos**, se mide la estabilidad del top 10 (coincidencia y Jaccard) y del ranking completo (Spearman $\rho$, Kendall $\tau$), y se hace un **Monte Carlo** con 5 000 vectores de pesos $w\sim\text{Dirichlet}(1,1,1)$ (uniforme en el simplex).

In [ ]:
C = cond[["C1", "C2", "C3"]].values
CONFIGS = {"Base (0.40/0.35/0.25)": (0.40, 0.35, 0.25), "Equitativa (1/3 c/u)": (1/3, 1/3, 1/3),
           "Énfasis distancia (0.60/0.20/0.20)": (0.60, 0.20, 0.20), "Énfasis capacidad (0.20/0.60/0.20)": (0.20, 0.60, 0.20),
           "Énfasis saturación (0.20/0.20/0.60)": (0.20, 0.20, 0.60), "Sin saturación (0.55/0.45/0.00)": (0.55, 0.45, 0.00)}
iv = {k: C @ np.array(w) for k, w in CONFIGS.items()}
base = iv["Base (0.40/0.35/0.25)"]; top_base = set(np.argsort(-base)[:10]); q0 = pd.qcut(base, 5, labels=False)

filas = []
for k, v in iv.items():
    t = set(np.argsort(-v)[:10]); q = pd.qcut(v, 5, labels=False, duplicates="drop")
    filas.append({"Configuración": k, "Top-10 en común (de 10)": len(top_base & t), "Jaccard top-10": len(top_base & t) / len(top_base | t),
                  "Spearman ρ (254 condados)": spearmanr(base, v)[0], "Kendall τ": kendalltau(base, v)[0],
                  "Condados que cambian de quintil": int((q != q0).sum())})
display(pd.DataFrame(filas).round(3).set_index("Configuración"))

# Rango de cada condado del top-10 base en cada configuración
R = pd.DataFrame({k: pd.Series(v).rank(ascending=False, method="min") for k, v in iv.items()}).astype(int)
orden = sorted(top_base, key=lambda i: -base[i])
display(R.loc[orden].assign(Condado=cond.loc[orden, "nombre"].values).set_index("Condado"))

In [ ]:
# Monte Carlo sobre el simplex de pesos
rng = np.random.default_rng(42); N = 5000
cnt = np.zeros(len(cond))
for w in rng.dirichlet([1, 1, 1], N):
    cnt[np.argsort(-(C @ w))[:10]] += 1
freq = pd.Series(cnt / N, index=cond["nombre"]).sort_values(ascending=False)
print(f"Condados que alguna vez entran al top-10 en {N} configuraciones: {(cnt > 0).sum()} de {len(cond)}")

# Sensibilidad a la REGLA DE IMPUTACIÓN (sin hospital -> máximo vs. percentil), distinta de los pesos
inv_obs = inv.copy(); obs = inv_obs[tiene]
filas = []
for p in (1.0, 0.90, 0.75):
    s = inv.copy(); s[~tiene] = obs.max() if p == 1.0 else obs.quantile(p)
    v = W_BASE[0]*cond["C1"] + W_BASE[1]*minmax(s) + W_BASE[2]*cond["C3"]
    t = list(np.argsort(-v.values)[:10])
    filas.append({"Imputación C2 (sin hospital)": "máximo" if p == 1.0 else f"percentil {int(p*100)}", "Top-10 en común": len(set(t) & top_base),
                  "Entran": ", ".join(cond.loc[list(set(t) - top_base), "nombre"]) or "—"})
display(pd.DataFrame(filas))

fig, ax = plt.subplots(figsize=(9, 3.8), dpi=110)
freq.head(12)[::-1].plot.barh(ax=ax, color="#bd0026"); ax.set_xlabel("Frecuencia en el top-10 (5 000 vectores de pesos)")
ax.set_title("Estabilidad del top-10 de vulnerabilidad ante cambios de pesos"); plt.tight_layout(); plt.show()

**Lectura de resultados:**

* **El top 10 es perfectamente estable ante los pesos:** en las seis configuraciones coinciden los mismos 10 condados *y en el mismo orden*, y en 5 000 pesos aleatorios solo 11 condados llegan alguna vez al top 10 (nueve de ellos en el 100 % de las simulaciones). **Pero esto no es “robustez sustantiva”, es una consecuencia de la imputación:** los 10 condados tienen $C_2=C_3=1$ (no tienen hospital ni hospital a 50 km con datos), así que $I_v=w_1C_1+(w_2+w_3)$ y su orden queda determinado **solo por $C_1$ (distancia)** para cualquier pesos. La estabilidad del top 10 mide la dominancia de la distancia entre los condados sin hospital, no la solidez del índice.
* **El resto del ranking sí es sensible:** Spearman cae a 0.93 (énfasis saturación) y 0.80 (sin saturación), Kendall a 0.62; entre 28 y 140 de los 254 condados cambian de quintil según la configuración, y hasta 29 saltan **más de un quintil** al eliminar $C_3$.
* **La regla de imputación pesa más que los pesos:** al imputar el percentil 90 o 75 en lugar del máximo para condados sin hospital, **2 de los 10 condados cambian** (entran Bastrop y Brewster).

**Implicación para política:** las recomendaciones del tipo “los 10 condados prioritarios para intervención de acceso” son robustas ante la elección de pesos y pueden comunicarse con confianza *condicionadas a la regla de imputación*; en cambio, recomendaciones que usan el quintil completo (focalización amplia, asignación proporcional de presupuesto) **no son robustas** y deberían presentarse como rangos, usar varios esquemas de pesos (o pesos elicitados con expertos, AHP) y complementarse con variables de demanda (edad, pobreza, seguro).

### 3.3b Caso adverso del algoritmo voraz (MCLP)

El voraz elige en cada paso el candidato de mayor ganancia **marginal**, sin ver el efecto conjunto. Como $f(S)=\sum_i a_i\,Y_i(S)$ es monótona y submodular, el voraz garantiza (Nemhauser et al., 1978)
$$f(S_{\text{greedy}})\ \ge\ \Big[1-\big(1-\tfrac1p\big)^{p}\Big]\,f(S^*)\ \ge\ \Big(1-\tfrac1e\Big)f(S^*)\approx0.632\,f(S^*),$$
y la cota es ajustada. **Configuración espacial adversa:** pares de núcleos de demanda separados aproximadamente por $2R$ con un candidato **equidistante** (en el punto medio) cuyo disco cubre la mitad de cada núcleo. Ese candidato centrado tiene la mayor ganancia individual (cubre dos mitades), el voraz lo elige primero y, a partir de ahí, cada candidato "natural" (centrado en un núcleo) ya solo aporta la mitad que quedó libre. El óptimo habría elegido los dos candidatos de los extremos.

Ejemplo (1D, $R=1$, $p=2$): demanda $a=5$ en $x=-1.8$, $b=6$ en $-0.8$, $c=6$ en $0.8$, $d=5$ en $1.8$; candidatos $S_1=-1.3$, $S_2=+1.3$, $S_3=0$.

In [ ]:
from itertools import combinations
R_, p_ = 1.0, 2
dem = {"a": (-1.8, 5), "b": (-0.8, 6), "c": (0.8, 6), "d": (1.8, 5)}
cand = {"S1": -1.3, "S2": 1.3, "S3": 0.0}
cubre = {s: {k for k, (x, _) in dem.items() if abs(x - xs) <= R_} for s, xs in cand.items()}
valor = lambda S: sum(dem[k][1] for k in set().union(*[cubre[s] for s in S]))

# voraz
S, cub = [], set()
for _ in range(p_):
    mejor = max((s for s in cand if s not in S), key=lambda s: sum(dem[k][1] for k in cubre[s] - cub))
    S.append(mejor); cub |= cubre[mejor]
# óptimo por fuerza bruta
opt = max(combinations(cand, p_), key=valor)
print(f"Voraz: {S} -> {valor(S)} | Óptimo: {opt} -> {valor(opt)} | razón = {valor(S)/valor(opt):.3f} (cota teórica p=2: {1-(1-1/p_)**p_:.3f})")

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 4.2), dpi=110, gridspec_kw={"width_ratios": [2.3, 1]})
col = {"S1": "#1b9e77", "S2": "#1b9e77", "S3": "#d95f02"}
for s, xs in cand.items():
    ax.add_patch(plt.Circle((xs, 0), R_, fill=(s == "S3"), alpha=.18 if s == "S3" else 1, ec=col[s], fc=col[s], lw=2.2, ls="-" if s != "S3" else "--"))
    ax.plot(xs, 0, marker="*" if s != "S3" else "X", ms=16, color=col[s]); ax.text(xs, -1.35, s + (" (voraz #1)" if s == "S3" else " (óptimo)"), ha="center", color=col[s], fontsize=9)
for k, (x, a) in dem.items():
    ax.scatter(x, 0, s=a * 55, color="#08519c", zorder=4); ax.text(x, 0.22, f"{k}={a}", ha="center", fontsize=9)
ax.set_xlim(-3.2, 3.2); ax.set_ylim(-1.6, 1.6); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Voraz: S3 cubre b+c = 12 → luego +5 = {valor(S)}  |  Óptimo: S1+S2 cubren a+b+c+d = {valor(opt)}", fontsize=10)
ps = np.arange(1, 21); ax2.plot(ps, 1 - (1 - 1 / ps) ** ps, "o-", color="#bd0026"); ax2.axhline(1 - 1 / np.e, ls="--", color="gray")
ax2.set_xlabel("p (instalaciones)"); ax2.set_ylabel("Cota del voraz"); ax2.set_title("Garantía 1−(1−1/p)ᵖ → 1−1/e"); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

**En Texas** este patrón puede aparecer con localidades pequeñas en fila a lo largo de una misma autopista rural (Trans-Pecos, Panhandle) separadas por ≈ $2R=50$ km: un candidato de la grilla de 50 km que cae justo entre dos localidades captura dos “mitades” y gana el primer paso, aunque dos candidatos sobre los núcleos habrían cubierto ambas localidades enteras. Además, la grilla de 50 km restringe el espacio de candidatos, por lo que la solución del Task 2.3 podría ser subóptima por dos razones: la miopía del voraz y la discretización (no se calculó el óptimo).

### 3.3c Tres limitaciones a comunicar al ministerio de salud de Texas

| # | Limitación | Por qué importa para decidir | Dato o mejora que la reduciría |
|---|---|---|---|
| **1** | **El acceso se mide con distancia/buffers y población uniforme por condado** (agregación en 254 polígonos). El análisis 3.1 muestra que el buffer puede sobre- o subestimar según la red vial, y la población de un condado extenso no está distribuida uniformemente. | Puede declarar “cubiertos” a habitantes que están a >15 min o ignorar a quienes están junto a una autopista; sesga la priorización y el MCLP. | Isocronas sobre la red vial completa del estado (OSM/TxDOT) con velocidades y tráfico reales por franja horaria, y población en malla fina (WorldPop 100 m, bloques censales). Validar con tiempos de ambulancia reales. |
| **2** | **Calidad y definición de la oferta:** se descartó el ~15 % de hospitales sin `camas_total`, la ocupación tiene faltantes (alimenta $C_3$), todos los tipos de hospital pesan igual (psiquiátrico, rehabilitación, general, CAH) y no se incluyen hospitales de estados vecinos/México (efecto de borde). Un hospital “cubre” aunque no tenga urgencias, UCI o personal. | Sobreestima la cobertura efectiva de emergencias y subestima la de condados fronterizos; el índice mezcla capacidad con saturación mal medida. | Inventario de servicios (urgencias 24/7, trauma, obstetricia, UCI) y personal por hospital, ocupación observada, hospitales fuera de Texas dentro de un margen de 50–100 km y métodos que ponderen capacidad (2SFCA). |
| **3** | **El índice de vulnerabilidad es normativo y frágil:** pesos subjetivos, imputación del máximo para condados sin hospital (que *determina* el top 10), y solo variables de oferta/distancia, sin demanda (edad, pobreza, seguro, carga de enfermedad). | La sensibilidad (3.3a) muestra top-10 estable pero quintiles inestables y dependientes de la imputación; focalizar presupuesto con quintiles no es defendible. | Pesos elicitados con expertos (AHP/Delphi) o derivados por PCA, incorporar variables de demanda (p. ej. índices de vulnerabilidad social del CDC, áreas médicamente desatendidas de HRSA) e imputar con modelos en lugar de extremos; reportar rangos de incertidumbre. |

**Mensaje al ministerio:** los resultados identifican con robustez *dónde* está el problema más extremo (condados sin hospital del oeste y sur), pero no deben usarse tal cual para dimensionar inversiones ni para cuantificar la población “cubierta”: falta tiempo de viaje real, servicios hospitalarios clave y variables de demanda.

---
## Referencias
* Church, R., & ReVelle, C. (1974). The maximal covering location problem. *Papers of the Regional Science Association, 32*(1), 101–118.
* Mutono, N., Wright, J. A., Mutunga, M., Mutembei, H., & Thumbi, S. M. (2022). Impact of traffic congestion on spatial access to healthcare services in Nairobi. *Frontiers in Health Services, 2*, Article 788173. https://doi.org/10.3389/frhs.2022.788173
* Nemhauser, G. L., Wolsey, L. A., & Fisher, M. L. (1978). An analysis of approximations for maximizing submodular set functions—I. *Mathematical Programming, 14*(1), 265–294.
* Boeing, G. (2017). OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks. *Computers, Environment and Urban Systems, 65*, 126–139.